## About Dataset
##### Context This is a small subset of dataset of Book reviews from Amazon Kindle Store category.

##### Content 5-core dataset of product reviews from Amazon Kindle Store category from May 1996 - July 2014. Contains total of 982619 entries. Each reviewer has at least 5 reviews and each product has at least 5 reviews in this dataset. Columns

1. asin - ID of the product, like B000FA64PK
2. helpful - helpfulness rating of the review - example: 2/3.
3. overall - rating of the product.
4. reviewText - text of the review (heading).
5. reviewTime - time of the review (raw).
6. reviewerID - ID of the reviewer, like A3SPTOKDG7WBLN
7. reviewerName - name of the reviewer.
8. summary - summary of the review (description).
9. unixReviewTime - unix timestamp.
10. Acknowledgements This dataset is taken from Amazon product data, Julian McAuley, UCSD website. http://jmcauley.ucsd.edu/data/amazon/

##### License to the data files belong to them.

### Inspiration

1. Sentiment analysis on reviews.
2. Understanding how people rate usefulness of a review/ What factors influence helpfulness of a review.
3. Fake reviews/ outliers.
4. Best rated product IDs, or similarity between products based on reviews alone (not the best idea ikr).
5. Any other interesting analysis

### Best Practises
1. Preprocessing And Cleaning
2. Train Test Split
3. BOW,TFIDF,Word2vec
4. Train ML algorithms

In [1]:
# Load the dataset
import pandas as pd
data=pd.read_csv('all_kindle_review.csv')
data.head()

,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


In [2]:
df=data[['reviewText','rating']]
df.head()

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",3
1,Great short read. I didn't want to put it dow...,5
2,I'll start by saying this is the first of four...,3
3,Aggie is Angela Lansbury who carries pocketboo...,3
4,I did not expect this type of book to be in li...,4


In [3]:
df.shape

(12000, 2)

In [4]:
## Missing Values
df.isnull().sum()

reviewText    0
rating        0
dtype: int64

In [5]:
df['rating'].unique()

array([3, 5, 4, 2, 1])

In [6]:
df['rating'].value_counts()

rating
5    3000
4    3000
3    2000
2    2000
1    2000
Name: count, dtype: int64

In [7]:
## Preprocessing And Cleaning
## postive review is 1 and negative review is 0
df['rating']=df['rating'].apply(lambda x:0 if x<3 else 1)

C:\Users\Gaurish16\AppData\Local\Temp\ipykernel_3580\3939683566.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['rating']=df['rating'].apply(lambda x:0 if x<3 else 1)


In [8]:
df['rating'].value_counts()

rating
1    8000
0    4000
Name: count, dtype: int64

In [9]:
## 1. Lower All the cases
df['reviewText']=df['reviewText'].str.lower()

C:\Users\Gaurish16\AppData\Local\Temp\ipykernel_3580\1654308119.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['reviewText']=df['reviewText'].str.lower()


In [10]:
df.head()

,reviewText,rating
0,"jace rankin may be short, but he's nothing to ...",1
1,great short read. i didn't want to put it dow...,1
2,i'll start by saying this is the first of four...,1
3,aggie is angela lansbury who carries pocketboo...,1
4,i did not expect this type of book to be in li...,1


In [11]:
import re
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Gaurish16\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [12]:
from bs4 import BeautifulSoup

In [17]:
pip install lxml

   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ------- -------------------------------- 0.8/4.0 MB 8.5 MB/s eta 0:00:01
   ---------- ----------------------------- 1.0/4.0 MB 3.6 MB/s eta 0:00:01
   --------------- ------------------------ 1.6/4.0 MB 3.5 MB/s eta 0:00:01
   ------------------ --------------------- 1.8/4.0 MB 3.0 MB/s eta 0:00:01
   ---------------------------------- ----- 3.4/4.0 MB 3.7 MB/s eta 0:00:01
   ---------------------------------------- 4.0/4.0 MB 3.6 MB/s  0:00:01
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
from nltk.corpus import stopwords
from bs4 import BeautifulSoup
import re

# Create stopwords set once
stop_words = set(stopwords.words('english'))

# Compile regex once
special_chars = re.compile(r'[^A-Za-z0-9 ]+')
url_pattern = re.compile(r'https?://\S+|www\.\S+')

def clean_text(text):
    text = str(text)

    # Remove HTML tags
    text = BeautifulSoup(text, "html.parser").get_text()

    # Remove URLs
    text = url_pattern.sub('', text)

    # Remove special characters
    text = special_chars.sub(' ', text)

    # Remove stopwords
    text = ' '.join(
        word for word in text.split()
        if word.lower() not in stop_words
    )

    # Remove extra spaces
    text = ' '.join(text.split())

    return text

# Avoid SettingWithCopyWarning
df = df.copy()

# Process all reviews in one pass
df.loc[:, 'reviewText'] = df['reviewText'].apply(clean_text)

In [20]:
df.head()

,reviewText,rating
0,jace rankin may short hes nothing mess man hau...,1
1,great short read didnt want put read one sitti...,1
2,ill start saying first four books wasnt expect...,1
3,aggie angela lansbury carries pocketbooks inst...,1
4,expect type book library pleased find price right,1


In [21]:
## Lemmatizer
from nltk.stem import WordNetLemmatizer

In [22]:
lemmatizer=WordNetLemmatizer()

In [23]:
def lemmatize_words(text):
    return " ".join([lemmatizer.lemmatize(word) for word in text.split()])

In [24]:
df['reviewText']=df['reviewText'].apply(lambda x:lemmatize_words(x))

In [25]:
df.head()

,reviewText,rating
0,jace rankin may short he nothing mess man haul...,1
1,great short read didnt want put read one sitti...,1
2,ill start saying first four book wasnt expecti...,1
3,aggie angela lansbury carry pocketbook instead...,1
4,expect type book library pleased find price right,1


In [26]:
## Train Test Split
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(df['reviewText'],df['rating'],
                                              test_size=0.20)

In [27]:

from sklearn.feature_extraction.text import CountVectorizer
bow=CountVectorizer()
X_train_bow=bow.fit_transform(X_train).toarray()
X_test_bow=bow.transform(X_test).toarray()

In [28]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf=TfidfVectorizer()
X_train_tfidf=tfidf.fit_transform(X_train).toarray()
X_test_tfidf=tfidf.transform(X_test).toarray()

In [29]:
X_train_bow

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(9600, 35823))

In [30]:
from sklearn.naive_bayes import GaussianNB
nb_model_bow=GaussianNB().fit(X_train_bow,y_train)
nb_model_tfidf=GaussianNB().fit(X_train_tfidf,y_train)

In [31]:

from sklearn.metrics import confusion_matrix,accuracy_score,classification_report

In [32]:
y_pred_bow=nb_model_bow.predict(X_test_bow)

In [33]:
y_pred_tfidf=nb_model_bow.predict(X_test_tfidf)

In [34]:
confusion_matrix(y_test,y_pred_bow)

array([[518, 275],
       [762, 845]])

In [35]:
print("BOW accuracy: ",accuracy_score(y_test,y_pred_bow))

BOW accuracy:  0.5679166666666666


In [36]:
confusion_matrix(y_test,y_pred_tfidf)

array([[512, 281],
       [759, 848]])

In [37]:
print("TFIDF accuracy: ",accuracy_score(y_test,y_pred_tfidf))

TFIDF accuracy:  0.5666666666666667
